In [ ]:
from google.colab import files

uploaded = files.upload()

Saving p1_behavior_test.csv to p1_behavior_test.csv
Saving p1_behavior_train.csv to p1_behavior_train.csv
Saving p1_behavior_val.csv to p1_behavior_val.csv


In [ ]:
import pandas as pd

train = pd.read_csv('/content/p1_behavior_train.csv')
val = pd.read_csv('/content/p1_behavior_val.csv')
test = pd.read_csv('/content/p1_behavior_test.csv')

print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)

print(train.head())

Train: (8000, 9)
Val: (1000, 9)
Test: (1000, 9)
                      learner_id concept_id assessment_id  mastery_score  \
0  KUK05_MAIN_ECE_SEC02_RISK_478     CON_OS       ASM_510         0.1528   
1  KUK05_MAIN_ECE_SEC02_RISK_758   CON_DBMS       ASM_515         0.3149   
2  KUK05_MAIN_ECE_SEC02_RISK_944    CON_DSA       ASM_530         0.4744   
3   KLU01_VJA_CSE_SEC01_RISK_116     CON_OS       ASM_590         0.4861   
4   NRI01_AP_CSE_SEC01_TRACK_383     CON_OS       ASM_589         0.9011   

   decay_rate  difficulty  response_correct     bloom_tier         timestamp  
0      0.0560      0.7805                 0    L1_Remember  05-06-2026 13:00  
1      0.0863      0.7416                 0    L1_Remember  17-06-2026 09:00  
2      0.0506      0.7774                 1    L1_Remember  05-06-2026 09:00  
3      0.0742      0.8083                 0  L2_Understand  10-06-2026 16:00  
4      0.0139      0.3417                 1     L4_Analyze  16-06-2026 12:00  


In [ ]:
print(train['mastery_score'].describe())

count    8000.000000
mean        0.459419
std         0.241920
min         0.150000
25%         0.271100
50%         0.395600
75%         0.518425
max         0.979600
Name: mastery_score, dtype: float64


In [ ]:
print(train.isnull().sum())

learner_id          0
concept_id          0
assessment_id       0
mastery_score       0
decay_rate          0
difficulty          0
response_correct    0
bloom_tier          0
timestamp           0
dtype: int64


In [ ]:
#Encoding test col
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    'concept_id',
    'assessment_id',
    'bloom_tier'
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()

    train[col] = le.fit_transform(train[col])

    val[col] = le.transform(val[col])

    test[col] = le.transform(test[col])

    encoders[col] = le

In [ ]:
features = [
    'decay_rate',
    'difficulty',
    'response_correct',
    'concept_id',
    'assessment_id',
    'bloom_tier'
]

X_train = train[features]
y_train = train['mastery_score']

X_val = val[features]
y_val = val['mastery_score']

X_test = test[features]
y_test = test['mastery_score']

In [ ]:
#train rf regressor
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

print("Training Completed")

Training Completed


In [ ]:
#validation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pred_val = rf.predict(X_val)

mae = mean_absolute_error(y_val, pred_val)

mse = mean_squared_error(y_val, pred_val)

rmse = mse ** 0.5

r2 = r2_score(y_val, pred_val)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.08924584099999998
RMSE: 0.10491763326740886
R2: 0.8130420258328512


In [ ]:
pred_test = rf.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)

mse = mean_squared_error(y_test, pred_test)

rmse = mse ** 0.5

r2 = r2_score(y_test, pred_test)

print("TEST RESULTS")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

TEST RESULTS
MAE: 0.08882416350000001
RMSE: 0.10397600202965468
R2: 0.8183145500840041


In [ ]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
})

print(
    importance.sort_values(
        by='Importance',
        ascending=False
    )
)

            Feature  Importance
5        bloom_tier    0.418520
0        decay_rate    0.336343
1        difficulty    0.185600
4     assessment_id    0.042122
3        concept_id    0.012595
2  response_correct    0.004819


In [ ]:
#testing
# XGBoost Regressor

!pip install xgboost -q

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

xgb_reg = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_reg.fit(X_train, y_train)

pred_test = xgb_reg.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
rmse = mean_squared_error(y_test, pred_test) ** 0.5
r2 = r2_score(y_test, pred_test)

print("XGBoost Regressor Results")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

XGBoost Regressor Results
MAE: 0.08839007444024086
RMSE: 0.10292112811920424
R2: 0.8219823778837128


In [ ]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance': xgb_reg.feature_importances_
})

print(
    importance.sort_values(
        by='Importance',
        ascending=False
    )
)

            Feature  Importance
5        bloom_tier    0.812639
1        difficulty    0.089291
0        decay_rate    0.075586
4     assessment_id    0.008381
2  response_correct    0.007712
3        concept_id    0.006391


In [ ]:
from sklearn.metrics import r2_score

train_pred = xgb_reg.predict(X_train)

train_r2 = r2_score(y_train, train_pred)

print("Train R2:", train_r2)

Train R2: 0.8821315961337589


In [ ]:
import joblib

joblib.dump(
    xgb_reg,
    "p1_behavior_xgboost_regressor.pkl"
)

print("Model Saved")

Model Saved
